# A11 · CVR、CTCVR、延迟与选择偏差

先修：公用31–34；A10。

证据类型：**明确构造机制实验（非真实广告日志）**。本章不声称工业系统复现或线上收益。

学习顺序：先预测 → 只改一个变量 → 运行 → 解释结果。

## 问题与输入输出契约

若转化定义为点击后的转化，P(conv|exposure)=P(click|exposure)×P(conv|click,exposure)。CVR训练只见点击样本，其特征分布可能与全曝光不同；不能在未点击样本上声称有CVR真值。延迟窗口未成熟时，未观察到的转化不一定是负例。下面构造10条曝光、4次点击、2次最终转化，不从Criteo click派生真实conversion。

运行前预测：本章只改变的变量会使哪一项结果变化？哪些不变量必须保持？

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'reclab').exists(): root=root.parent
if str(root) not in sys.path: sys.path.insert(0,str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reclab import ads_curriculum as ac

## 运行单变量对照

输入是源码中可见的小型构造数据，输出单位和约束按上文定义。运行断言核验数学与账本。

In [2]:
early=ac.conversion_delay(2);mature=ac.conversion_delay(7)
display(pd.DataFrame([early,mature],index=['2 days','7 days']))
assert early['observed_cvr']<mature['observed_cvr']
assert early['clicked_high_share']!=early['exposure_high_share']
assert np.isclose(mature['ctr']*mature['eventual_cvr'],mature['eventual_ctcvr'])

,ctr,eventual_cvr,eventual_ctcvr,observed_cvr,unmatured_positive,clicked_high_share,exposure_high_share
2 days,0.4,0.5,0.2,0.25,1,0.75,0.3
7 days,0.4,0.5,0.2,0.50,0,0.75,0.3


## 解释与局限

窗口从2天改成7天时最终CTCVR不变、观测CVR升高。高响应层在点击样本占比更高，说明选择分布改变。联合CTR/CTCVR损失可以讲全空间约束，但没有真实转化日志不能做真实ESMM效果结论。

## 查看源码，再做练习

加入转化归因窗口与未转化的成熟样本；计算按cohort成熟度报告的CVR，而非把所有新曝光立即标零。

In [3]:
import inspect
print(inspect.getsource(ac.conversion_delay))

def conversion_delay(cutoff=2):
    # Each tuple is (clicked, eventual_conversion, delay_days, stratum).
    logs=[(1,1,1,'high'),(1,1,5,'high'),(1,0,0,'high'),(1,0,0,'low')]+[(0,0,0,'low')]*6
    clicks=sum(x[0] for x in logs); eventual=sum(x[1] for x in logs)
    observed=sum(c and y and d<=cutoff for c,y,d,_ in logs)
    return dict(ctr=clicks/len(logs),eventual_cvr=eventual/clicks,eventual_ctcvr=eventual/len(logs),
                observed_cvr=observed/clicks,unmatured_positive=eventual-observed,
                clicked_high_share=sum(c and s=='high' for c,_,_,s in logs)/clicks,
                exposure_high_share=sum(s=='high' for *_,s in logs)/len(logs))



## 来源与下一步

A-RTB response/attribution主题；延迟与选择偏差为Lab概率展开，并非原课件完整复现。这是Lab自行设计的小实验，不复制讲义全文。

[GitHub中文单章](https://github.com/SocratesClub/ccrbook/blob/6dbd5650f2adc6e257e5bd5d156bc50a72560ee9/slides/Chapter%208%20%E8%AE%A1%E7%AE%97%E5%B9%BF%E5%91%8A.pptx) · [作者RTB讲义](https://wnzhang.net/slides/cikm-16-rtb-full.pdf)。中文PPT不是现代广告工程全套课；不将历史平台机制描述当作当前市场事实。

完成练习后回到广告学习路径，继续下一语义单元。